In [2]:
"""full_3.ipynb — train the Channel-flow (A1) model via the train_channel.py module.

This notebook is a thin wrapper around `train_channel.run_trial`: it sets the
config (here 3 epochs for a smoke test) and runs one trial, then reports the
result. The heavy lifting (data loading into RAM, adaptive LR, best-checkpoint
S3 save, Trackio logging) all lives in `train_channel.py`.

Run this cell first to make sure the module imports cleanly.
"""
import sys
from pathlib import Path


from train_channel import (
    run_trial,
    EPOCHS,
    BATCH_SIZE,
    NUM_WORKERS,
    S3_MODEL_PREFIX,
    S3_BUCKET,
    STORE_PATH,
)
print("train_channel imported OK")
print(f"  default EPOCHS   = {EPOCHS}")
print(f"  BATCH_SIZE       = {BATCH_SIZE}")
print(f"  NUM_WORKERS      = {NUM_WORKERS}")
print(f"  S3 dataset       = s3://{S3_BUCKET}/{STORE_PATH}")
print(f"  S3 model prefix  = {S3_MODEL_PREFIX}")


ModuleNotFoundError: No module named 'train_channel'

# Couette — per-instance prediction analysis

Load a single trained checkpoint, generate predictions on the validation set, and study **instance-level** behavior:

1. **Loss per instance** — is the model failing on some snapshots more than others?
2. **Instance reconstruction in 3D** — visualize original / reconstruction / error as volumetric fields (pyvista, notebook mode).

Edit `S3_CHECKPOINT` + `WIDEN` in the load cell to pick the model.


In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda"
keys = 'u', 'v', 'w', 'p_total'
comp_labels = ['u_x', 'u_y', 'u_z']

def extract(real, keys):
    with h5py.File("/data1/dataToYouri/couette_dns_ml_f32.h5", 'r') as f:
        return np.stack([np.array(f[real][k]) for k in keys], axis=1)

realizations = ['R13','R14','R15','R18','R19','R21','R22','R23','R26','R28','R3','R4','R5']
samplings = {r: (np.s_[:-1:5000//300] if r in ["R3","R4","R5"] else np.s_[:]) for r in realizations}
raw_d = {r: extract(r, keys)[samplings[r]] for r in realizations}
all_data = np.concatenate(list(raw_d.values()), axis=0)

val_realizations = ["R18", "R5"]
val_npds = np.concatenate([raw_d[r] for r in val_realizations])

with h5py.File("/data1/dataToYouri/couette_dns_ml_f32.h5", 'r') as f:
    y = np.array(f["y"]); x = np.array(f["x"]); z = np.array(f["z"])

# track which val sample came from which realization / original index (for the "instance" id)
realization_of = []
idx_within = []
for r in val_realizations:
    realization_of += [r] * raw_d[r].shape[0]
    idx_within += list(range(raw_d[r].shape[0]))
realization_of = np.array(realization_of)
idx_within = np.array(idx_within)

print("val:", val_npds.shape, "| realizations:", sorted(set(realization_of)))

# ---------------------------------------------------------------------------
# Normalization + dataset
# ---------------------------------------------------------------------------
naxes = (0, 2, 4)
mean = all_data.mean(naxes, keepdims=True)
std = all_data.std(naxes, keepdims=True) + 1e-10
mean_gpu = torch.from_numpy(mean).to(device)
std_gpu = torch.from_numpy(std).to(device)

def normalize(d): return (d - mean) / std
def unnormalize_velocity_gpu(d): return (d * std_gpu[:, :3]) + mean_gpu[:, :3]

class DS(Dataset):
    def __init__(self, d):
        d = normalize(d)
        self.x = torch.from_numpy(d[:, 3:, :, 0, :]).float().to(device)
        self.y = torch.from_numpy(d[:, :3, :, :, :]).float().to(device)
    def __len__(self): return self.x.shape[0]
    def __getitem__(self, i): return self.x[i], self.y[i]

val_ds = DS(val_npds)
val_dl = DataLoader(val_ds, batch_size=4, shuffle=False, drop_last=True)
target_phys = val_npds[:, :3, :, :, :].astype(np.float32)
print(f"val_ds: {len(val_ds)} samples, {len(val_dl)} batches")


In [ ]:
# =========================================================================
# Load ONE checkpoint + generate per-instance predictions
# =========================================================================
S3_BUCKET = "simulations"
S3_ENDPOINT = "http://localhost:8333"
S3_CHECKPOINT = "flow_models/couette/width-study_lw32_c32_b10_w0/final.pt"   # <-- pick your model
WIDEN = 0                                                        # <-- 0 / 1 / 32 / 64

import s3fs, io
fs = s3fs.S3FileSystem(profile='default', client_kwargs={'endpoint_url': S3_ENDPOINT})
buf = fs.cat_file(f"{S3_BUCKET}/{S3_CHECKPOINT}")
ckpt = torch.load(io.BytesIO(buf), map_location="cpu")
cfg = dict(ckpt["config"])
sd = ckpt["model_state_dict"]
widen = int(cfg.get("widen", 0)) or (int(sd["pre_widen.0.weight"].shape[0]) if "pre_widen.0.weight" in sd else 0)
print("config:", cfg, " widen (effective):", widen)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv3d(in_ch, out_ch, 3, padding="same"), nn.SiLU(),
                                  nn.Conv3d(out_ch, out_ch, 3, padding="same"), nn.SiLU())
    def forward(self, x): return self.conv(x)

class M4(nn.Module):
    D, XD, ZD = 33, 24, 24
    def __init__(self, lift_width=32, c3d=32, n_blocks=5, widen=0):
        super().__init__()
        self.pressure_encoder = nn.Sequential(nn.Conv2d(1,8,3,padding="same"), nn.SiLU(),
                                              nn.Conv2d(8,32,3,padding="same"), nn.SiLU())
        in_lift = 32
        if widen > 0:
            self.pre_widen = nn.Sequential(nn.Conv2d(32, widen, 3, padding="same"), nn.SiLU())
            in_lift = widen
        self.lift = nn.Conv2d(in_lift, self.D*lift_width, 3, padding="same")
        self.lift_proj = (nn.Conv3d(lift_width, c3d, 1) if lift_width != c3d else nn.Identity())
        self.conv_blocks = nn.Sequential(*[ConvBlock(c3d, c3d) for _ in range(n_blocks)])
        self.output = nn.Conv3d(c3d, 3, 3, padding="same")
    def forward(self, pressure):
        x = self.pressure_encoder(pressure)
        if hasattr(self, "pre_widen"): x = self.pre_widen(x)
        x = self.lift(x)
        x = x.reshape(-1, self.lift.out_channels // self.D, self.XD, self.D, self.ZD)
        x = self.lift_proj(x)
        x = self.conv_blocks(x)
        return self.output(x)

model = M4(lift_width=cfg["lift_width"], c3d=cfg["c3d"], n_blocks=cfg["n_blocks"], widen=widen).to(device)
model.load_state_dict(sd)
model.eval()
print(f"loaded {sum(p.numel() for p in model.parameters())/1e6:.2f}M params")

# ---- predict (per-instance, in physical units) ----
@torch.no_grad()
def predict_phys(ds):
    out = [model(x) for x, _ in ds]
    p = torch.cat(out, 0)
    return unnormalize_velocity_gpu(p).cpu().numpy()

pred_phys = predict_phys(val_dl)          # (N, 3, X, Y, Z) physical units
print("pred:", pred_phys.shape, "target:", target_phys.shape)
print("pred range [%.3f, %.3f]" % (pred_phys.min(), pred_phys.max()))


In [ ]:
# =========================================================================
# 1) Loss per instance — which validation snapshots fail more than others?
# =========================================================================
diff_phys = pred_phys - target_phys                       # (N, 3, X, Y, Z)
mse_inst = (diff_phys ** 2).mean(axis=(1, 2, 3, 4))       # (N,)  per-instance MSE

inst_names = np.array([f"{realization_of[i]}#{idx_within[i]}" for i in range(len(mse_inst))])

print("per-instance MSE:  min %.2e  median %.2e  mean %.2e  max %.2e" %
      (mse_inst.min(), np.median(mse_inst), mse_inst.mean(), mse_inst.max()))

# per-component per-instance mse
mse_comp = (diff_phys ** 2).mean(axis=(2, 3, 4))          # (N, 3)
for c, l in enumerate(comp_labels):
    print(f"  {l}: mean {mse_comp[:, c].mean():.3e}  max {mse_comp[:, c].max():.3e}")

# ---- panel 1: distribution of per-instance error (log scale) ----
order = np.argsort(mse_inst)
fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))

ax[0].hist(mse_inst, bins=40)
ax[0].set_yscale("log")
ax[0].set_title("Per-instance MSE (log y)")
ax[0].set_xlabel("MSE")
ax[0].set_ylabel("count")
ax[0].axvline(np.median(mse_inst), color="k", ls="--", label=f"median {np.median(mse_inst):.1e}")
ax[0].legend()

ax[1].plot(np.log10(mse_inst[order]))
ax[1].set_title("MSE by rank (worst → best)")
ax[1].set_xlabel("instance rank")
ax[1].set_ylabel("log10 MSE")
ax[1].grid(True, alpha=0.3)

# per component, box plot
ax[2].boxplot(mse_comp, labels=comp_labels, showfliers=False)
ax[2].set_title("Per-component MSE per instance (log)")
ax[2].set_yscale("log")
ax[2].set_ylabel("MSE")

plt.tight_layout()
plt.show()

# ---- worst / best instances ----
print("\nWORST 5 instances:")
for i in order[-5:][::-1]:
    print(f"  {inst_names[i]:<12s}  mse={mse_inst[i]:.3e}   (u:{mse_comp[i,0]:.2e} v:{mse_comp[i,1]:.2e} w:{mse_comp[i,2]:.2e})")
print("\nBEST 5 instances:")
for i in order[:5]:
    print(f"  {inst_names[i]:<12s}  mse={mse_inst[i]:.3e}   (u:{mse_comp[i,0]:.2e} v:{mse_comp[i,1]:.2e} w:{mse_comp[i,2]:.2e})")

# how much does error vary between realizations?
for r in sorted(set(realization_of)):
    m = mse_inst[realization_of == r]
    print(f"  {r}: n={len(m)}  mean mse={m.mean():.3e}  median={np.median(m):.3e}")


In [ ]:
# =========================================================================
# 2) 3D reconstruction viewer -- PyVista volume of |u| magnitude.
#    StructuredGrid from the real (x, y, z) grid points (y is CGL, non-uniform).
#    Controls live INSIDE the trame app (server.state + change callbacks) --
#    ipywidgets can't reach the trame scene, so we don't use them.
#    Only two controls: field toggle + time slider.
# =========================================================================
import pyvista as pv
import numpy as np
from trame.widgets import vuetify3 as vuetify
from trame.app import get_server

pv.set_jupyter_backend('trame')
pv.global_theme.trame.jupyter_server_port = 36167   # stable port

N, C, X, Y, Z = target_phys.shape

# StructuredGrid from the actual grid points (y non-uniform CGL handled)
_Xg = np.empty((Z, Y, X)); _Xg[:] = x[None, None, :]
_Yg = np.empty((Z, Y, X)); _Yg[:] = y[None, :, None]
_Zg = np.empty((Z, Y, X)); _Zg[:] = z[:, None, None]
grid = pv.StructuredGrid(_Xg, _Yg, _Zg)

def _to_grid(arr):
    g = grid.copy(deep=True)
    g['vel'] = np.ascontiguousarray(arr.transpose(2, 1, 0)).ravel()
    g.set_active_scalars('vel')
    return g

# fields: |u| magnitude per snapshot (N, X, Y, Z)
F = {
    'original':  np.linalg.norm(target_phys, axis=1),
    'predicted': np.linalg.norm(pred_phys,   axis=1),
    'error':     np.linalg.norm(diff_phys,   axis=1),
}
cmax = {k: float(np.percentile(v, 95)) for k, v in F.items()}

pl = pv.Plotter(notebook=True, window_size=(720, 600))
pl.set_background('white')

# ---- trame state ----
K_FIELD, K_INST = 'cv_field', 'cv_inst'
server = get_server(pv.global_theme.trame.jupyter_server_name)

def _redraw(_=None):
    f = server.state[K_FIELD]; i = int(server.state[K_INST])
    pl.clear_actors(); pl.add_axes()
    pl.add_volume(_to_grid(F[f][i]), cmap='viridis', opacity=0.30, clim=(0, cmax[f]))
    pl.reset_camera()
    pl.add_scalar_bar(title=f'|u|  ({f}, t={i})', position_x=0.95)

server.state.change(K_FIELD)(_redraw)
server.state.change(K_INST)(_redraw)

# initial state
server.state[K_FIELD] = 'predicted'
server.state[K_INST]  = 0
_redraw()

# ---- controls (inside the trame app) ----
def _make_controls():
    row = vuetify.VRow(classes='pa-1 ma-0 align-center', style='gap:12px;')
    with row:
        vuetify.VBtnToggle(v_model=(K_FIELD, 'predicted'), buttons=[
            {'title': k, 'value': k} for k in F])
        vuetify.VSlider(v_model=(K_INST, 0), min=0, max=N-1, step=1, label='t')
        vuetify.VBtn(label='reset cam', on_click=lambda v: pl.reset_camera())

w = pl.show(jupyter_kwargs=dict(add_menu_items=_make_controls))
from IPython.display import display
display(w)
